El notebook está en `notebooks/` y el código del proyecto en `src/`. Para poder importarlo se agrega la carpeta raíz del proyecto (la de arriba) a la lista de lugares donde Python busca módulos. `sys.path` solo acepta texto, por eso se convierte con `str()`, y se agrega únicamente si todavía no está, para que volver a ejecutar la celda no la repita.

In [ ]:
import sys
from pathlib import Path

raiz = str(Path.cwd().parent)
if raiz not in sys.path:
    sys.path.append(raiz)

from src import config, data_loader, profiling

print("Raíz del proyecto:", raiz)
print("Carpeta de datos: ", config.DIR_DATOS)

Qué se analiza se define acá y no dentro de las funciones. Para analizar otro año, o varios, alcanza con cambiar `ANIOS` (por ejemplo `["2023", "2024"]`): el resto del notebook no se modifica.

**Modificación 1.** Los archivos de la EPH están fuera del proyecto, en una carpeta `files/` que está al lado de la carpeta del proyecto. Esa ubicación se configura en **una sola línea**: `DIR_DATOS` en `src/config.py`.

In [ ]:
CATEGORIA = config.CAT_MERCADO_LABORAL
TIPO = config.TIPO_HOGAR
ANIOS = ["2024"]

if not config.DIR_DATOS.exists():
    raise FileNotFoundError(
        f"No se encontró la carpeta de datos: {config.DIR_DATOS}. "
        "Debe llamarse 'files' y estar al lado de la carpeta del proyecto."
    )

El escáner recorre la carpeta de datos (en cualquier nivel de subcarpetas, porque los archivos se dejan como se descomprimen) y reconoce el tipo de archivo, el año y el trimestre por el **nombre** (por ejemplo, `usu_hogar_T124.txt` es hogares, 2024, trimestre 1). Así, agregar un año o una categoría nueva no requiere cambiar el código.

In [ ]:
todos = data_loader.escanear_datos()

resumen = {}
for archivo in todos:
    clave = (archivo["categoria"], archivo["anio"])
    if clave not in resumen:
        resumen[clave] = []
    if archivo["trimestre"] not in resumen[clave]:
        resumen[clave].append(archivo["trimestre"])

print("Archivos detectados:", len(todos))
for (categoria, anio), trimestres in sorted(resumen.items()):
    print(f"  {categoria:22} {anio}  {sorted(trimestres)}")

### Verificación de trimestres completos

Se necesitan los **4 trimestres** de cada año analizado, con el archivo de hogares y el de individuos. `trimestres_faltantes` lista los que no tengan los dos archivos. Si falta alguno se avisa, pero no se detiene el análisis: los trimestres que falten quedan registrados en el informe.

In [ ]:
faltantes_por_anio = {}
for anio in ANIOS:
    faltan = data_loader.trimestres_faltantes(todos, CATEGORIA, anio)
    faltantes_por_anio[anio] = faltan
    if faltan:
        print(f"ATENCIÓN {anio}: faltan los trimestres {faltan}")
    else:
        print(f"{anio}: los 4 trimestres están completos")

archivos = data_loader.buscar_archivos(
    tipo=TIPO, categoria=CATEGORIA, anios=ANIOS)
print(f"\nArchivos de hogares a analizar: {len(archivos)}")
for archivo in archivos:
    print("  ", archivo["anio"], archivo["trimestre"], archivo["ruta"].name)

Antes de analizar, un vistazo a un trimestre: los archivos son de texto separado por `;`, cada fila es un hogar y todos los valores se leen como **texto** (hay que convertirlos para hacer cuentas). Las filas se leen de a un trimestre por vez y se descartan al terminar, para no ocupar memoria de más.

In [ ]:
filas = data_loader.leer_filas(archivos[0]["ruta"])

print("Archivo:", archivos[0]["ruta"].name)
print("Hogares (filas):", len(filas))
print("Variables (columnas):", len(filas[0]))
print("Primeras variables:", list(filas[0])[:8])

print("\nEjemplo de un hogar:")
for columna in ["ANO4", "TRIMESTRE", "AGLOMERADO", "MAS_500", "ITF", "GDECCFR"]:
    print(f"  {columna:10} = {filas[0][columna]!r}")

del filas

Las variables salen de `config.VARS_HOGARES`, que combina tres grupos:

- las **pedidas por el enunciado** (`ANO4`, `TRIMESTRE`, `ITF`, `MAS_500`, `GDECCFR`);
- **dos elegidas por el grupo** (`REALIZADA` e `IV1`);
- las agregadas por el **anexo** (`CODUSU`, `PONDERA`, `AGLOMERADO`, `REGION`).

Criterio de elección de las dos variables del grupo: `REALIZADA` indica si la entrevista del hogar se realizó (sirve de control de la propia encuesta) e `IV1` es el tipo de vivienda, una variable de contexto del hogar. *(El grupo debe confirmar o ajustar este criterio en la bitácora.)*

Para cambiar el listado se modifican las listas de `config.py`. Abajo se muestran las reglas de faltantes que se aplican a cada variable, además de que un campo **vacío** siempre es faltante real.

In [ ]:
print("Variables de hogares analizadas:", len(config.VARS_HOGARES))
print("  Pedidas por el enunciado:", config.VARS_HOGARES_OBLIGATORIAS)
print("  Elegidas por el grupo:   ", config.VARS_HOGARES_EXTRA)
print("  Agregadas por el anexo:  ", config.VARS_HOGARES_ANEXO)

print("\nReglas especiales de faltantes (además del campo vacío):")
for columna in config.VARS_HOGARES:
    reales = config.FALTANTES_REALES.get(columna)
    no_corresponde = config.NO_CORRESPONDE.get(columna)
    if reales:
        print(f"  {columna}: los códigos {reales} cuentan como faltante real")
    if no_corresponde:
        print(f"  {columna}: los códigos {no_corresponde} son no corresponde")
print("  (el resto de las variables solo tiene la regla del campo vacío)")

Para cada archivo se cuentan, por variable, los valores **válidos**, los **faltantes reales** y los **no corresponde**. Se guardan los **conteos** (no los porcentajes) porque así después se pueden sumar los trimestres para obtener el año y aplicar distintos umbrales sin volver a leer los archivos.

In [ ]:
resultados = []
for archivo in archivos:
    filas = data_loader.leer_filas(archivo["ruta"])
    conteos = profiling.contar_valores(filas, config.VARS_HOGARES)
    resultados.append({
        "anio": archivo["anio"],
        "trimestre": archivo["trimestre"],
        "conteos": conteos,
    })
    print(f"{archivo['anio']}-{archivo['trimestre']}: "
          f"{conteos['total_filas']} hogares analizados")
    if conteos["inexistentes"]:
        print("  Variables que no están en el archivo:",
              conteos["inexistentes"])
    del filas

metadatos = {
    "categoria": CATEGORIA,
    "tipo_archivo": TIPO,
    "anios": ANIOS,
    "trimestres_faltantes": faltantes_por_anio,
}

Se muestra primero el umbral base de 5% (`UMBRAL_FALTANTES_PCT`). Una variable **supera el umbral** si su porcentaje de faltantes reales es **mayor** al umbral. El porcentaje se calcula sobre el total de hogares del trimestre.

In [ ]:
def mostrar_tabla(bloque):
    """Imprime la completitud de cada variable de un bloque de resultados."""
    print(f"{'Variable':12}{'Válidos':>9}{'Faltantes':>11}"
          f"{'No corresp.':>13}{'% faltantes':>13}  Supera")
    for columna, d in bloque["columnas"].items():
        marca = "SÍ" if d["supera_umbral"] else ""
        print(f"{columna:12}{d['validos']:>9}{d['faltantes_reales']:>11}"
              f"{d['no_corresponde']:>13}{d['pct_faltantes']:>13.2f}  {marca}")


informe_base = profiling.armar_informe_completitud(
    resultados, config.UMBRAL_FALTANTES_PCT, metadatos)

for periodo, bloque in informe_base["por_trimestre"].items():
    print(f"\n=== {periodo}: {bloque['total_filas']} hogares ===")
    mostrar_tabla(bloque)
    print("Superan el", config.UMBRAL_FALTANTES_PCT, "% de faltantes:",
          bloque["sobre_umbral"])
    print("Mayor porcentaje de faltantes:", bloque["mayor_faltante"],
          f"({bloque['pct_mayor_faltante']}%)")

El año se calcula **sumando los conteos de sus trimestres** y dividiendo una sola vez. No se promedian los porcentajes de cada trimestre, porque los trimestres no tienen exactamente la misma cantidad de hogares y el resultado cambiaría.

In [ ]:
for anio, bloque in informe_base["anual"].items():
    print(f"=== Año {anio} (trimestres {bloque['trimestres_incluidos']}): "
          f"{bloque['total_filas']} hogares ===")
    mostrar_tabla(bloque)
    print("Superan el", config.UMBRAL_FALTANTES_PCT, "% de faltantes:",
          bloque["sobre_umbral"])
    print("Mayor porcentaje de faltantes:", bloque["mayor_faltante"],
          f"({bloque['pct_mayor_faltante']}%)")

Los umbrales a comparar están en `config.UMBRALES_FALTANTES_PCT` (hoy 5, 10 y 20). Para agregar o cambiar un porcentaje se modifica esa lista: el informe de cada umbral se genera solo, sin repetir el análisis a mano. Los conteos ya están calculados, solo se vuelve a aplicar el umbral.

In [ ]:
for umbral in config.UMBRALES_FALTANTES_PCT:
    informe = profiling.armar_informe_completitud(
        resultados, umbral, metadatos)
    print(f"--- Umbral {umbral}% ---")
    for anio, bloque in informe["anual"].items():
        print(f"  Año {anio}: {bloque['sobre_umbral']}")
    for periodo, bloque in informe["por_trimestre"].items():
        print(f"  {periodo}: {bloque['sobre_umbral']}")

¿Por qué los tres umbrales dan las mismas variables? Porque `ITF` está alrededor de 22% a 26% de faltantes y `GDECCFR` alrededor de 53%: ambas superan 5%, 10% y 20%, y el resto de las variables tiene 0%. Para ver que el umbral **sí** cambia el resultado, se prueban otros valores sin tocar ningún archivo del proyecto:

In [ ]:
anual = informe_base["anual"]
for umbral in (25, 30, 60):
    informe = profiling.armar_informe_completitud(
        resultados, umbral, metadatos)
    print(f"Umbral {umbral}%:")
    for anio, bloque in informe["anual"].items():
        print(f"  año {anio}: {bloque['sobre_umbral']}")
    for periodo, bloque in informe["por_trimestre"].items():
        print(f"  {periodo}: {bloque['sobre_umbral']}")

Se genera un archivo por cada umbral, con el umbral en el nombre (`calidad_datos_umbral_5.json`, `..._10.json`, `..._20.json`) para identificarlos a simple vista. El nombre se arma con el patrón `ARCHIVO_CALIDAD` de `config.py`, y los archivos se guardan en la carpeta `output/`.

Cada archivo es un reporte que se va completando por requerimiento: este notebook escribe la sección `completitud_hogares` sin pisar las demás secciones que pueda haber guardado otro notebook.

In [ ]:
for umbral in config.UMBRALES_FALTANTES_PCT:
    informe = profiling.armar_informe_completitud(
        resultados, umbral, metadatos)
    ruta = config.DIR_SALIDA / config.ARCHIVO_CALIDAD.format(umbral=umbral)
    data_loader.guardar_reporte("completitud_hogares", informe, ruta)
    print("Guardado:", ruta)

En todos los trimestres la variable con mayor porcentaje de faltantes es **`GDECCFR`** (alrededor de 53%). Si hubiera más de una variable con la misma cantidad máxima de faltantes, se informan **todas**.

Ese porcentaje alto **no es un problema de calidad**. `GDECCFR` es el decil de ingreso de los aglomerados de 500.000 o más habitantes, así que debería estar vacío cuando el hogar es de un aglomerado chico (`MAS_500 = N`). Se comprueba cruzando las dos variables:

In [ ]:
print("Hogares por MAS_500 y si GDECCFR está vacío:\n")
for archivo in archivos:
    filas = data_loader.leer_filas(archivo["ruta"])
    cuenta = {}
    for fila in filas:
        clave = (fila["MAS_500"], fila["GDECCFR"] == "")
        cuenta[clave] = cuenta.get(clave, 0) + 1
    print(f"{archivo['anio']}-{archivo['trimestre']}:")
    print(f"   MAS_500 = N  y GDECCFR vacío:     {cuenta.get(('N', True), 0)}")
    print(f"   MAS_500 = N  y GDECCFR con valor: {cuenta.get(('N', False), 0)}")
    print(f"   MAS_500 = S  y GDECCFR vacío:     {cuenta.get(('S', True), 0)}")
    print(f"   MAS_500 = S  y GDECCFR con valor: {cuenta.get(('S', False), 0)}")
    del filas

Todos los vacíos de `GDECCFR` corresponden a hogares de aglomerados chicos y ningún hogar de aglomerados grandes lo tiene vacío. Por eso se informa el porcentaje sin ajustar (para que sea comparable con otros grupos) y se agrega una **nota** en el informe que lo explica (ver `notas` en los archivos JSON).

## 12. Conclusiones

*(Valores de 2024.)*

- Los 4 trimestres de hogares están completos: entre 16.104 y 16.650 hogares por trimestre, 65.718 en el año.
- Las variables de identificación y de periodo (`CODUSU`, `ANO4`, `TRIMESTRE`, `AGLOMERADO`, `REGION`, `PONDERA`), `MAS_500`, `REALIZADA` e `IV1` no tienen faltantes.
- `ITF` tiene entre 22,6% y 25,9% de faltantes por trimestre (24,2% en el año). Se interpreta un `0` como "no responde": 97,6% de esos hogares tiene el decil de ingreso en 12 (no respuesta de ingresos).
- `GDECCFR` tiene alrededor de 53% de vacíos, pero corresponden a "no corresponde" (aglomerados de menos de 500.000 habitantes).
- Con los umbrales de 5%, 10% y 20% se informan las mismas dos variables: `ITF` y `GDECCFR`.

**Decisiones y problemas encontrados:** ver la bitácora, sección A.1 (definición de faltante, el cambio de formato de INDEC entre T2 y T3, y el criterio de elección de las variables extra).